# The Hurst phenomenon: persistence, long-range correlations, self-similarity, and all that...

**Aim:**  
- Study the Hurst phenomenon (a Hurst exponent $H > 0.5$), also called persistence, on microstate sequences 

**Background:**  
Computing the Hurst exponent is the easy part and obtaining $H>0.5$ from biological signals is common. The difficult part is interpretation:
- Certain carefully designed stochastic processes can be constructed based on $H$ as the only parameter. A famous example is fractional Gaussian noise (fGn) and its integral (or random walk) fractional Brownian motion (fBm). These processes show *other* interesting properties such as long-range correlations (infinite integral over the autocorrelation function); self-similarity (but only in the *statistical* sense, *not strict* self-similarity in the literal sense); fractality. These properties are built into fGn/fBm by construction, but they don't come for free with an empirical Hurst exponent H>0.5 from real data.
- Finding a __Hurst exponent H>0.5 does not imply any of the aforementioned properties__, counterexamples abund.
- __Persistence__ is the most careful verbalization of H>0.5, and indicates autocorrelations _of any kind_  across several time scales
- The key to interpretation lies in the method used to calculate $H$. Two common methods are Detrended Fluctuation Analysis (DFA) and its wavelet analogue. Importantly, both only measure (detrended) **variance** at a given scale. There is no analysis as to whether:
  - the variance is generated by exactly the same mechanism at each scale,
  - the signal fulfills any similarity criteria across the scales (self-similarity) - variance **_is_** the similarity criterion
  - the marginal density at a single time point looks alike at different scales, in agreement with the Hurst exponent (density scaling) 
- Any process composed of different time scales is likely to yield $H>0.5$. Therefore, scale-rich processes are just as likely to yield $H>0.5$ as (some) scale-free processes. It is therefore not justified to infer scale-freeness from $H>0.5$ alone

Some of these thoughts will be further illustrated in this notebook.

**Method(s):**
- **Partiton method** (`mstsa.randomwalk` and `mstsa.dfa`): partition the available microstates into two classes and map each class to -1, +1, respectively; construct the random walk and compute the Hurst exponent via DFA 
  microstate's binary indicator sequence, returning one Hurst exponent per state.
- **Microstate characteristic (or indicator) functions** (`mstsa.hurst_exponents_dfa`): compute the binary indicator sequence for each microstate class, resulting in K binary sequences, construct the random walk and compute the Hurst exponent via DFA

**Data**: MPILMBB EEG microstate sequences

In [ ]:
# JupyterLite/Pyodide only: install the pure-Python mstsa build (no numba/C
# extensions -- see https://github.com/Frederic-vW/mstsa/tree/pyodide). Falls
# through silently on a normal Jupyter install, where mstsa is already present.
try:
    import piplite
    await piplite.install(
        "https://raw.githubusercontent.com/Frederic-vW/mstsa/pyodide/wheels/mstsa-0.4.3-py3-none-any.whl"
    )
except ImportError:
    pass


In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, shapiro, ttest_rel, wilcoxon
import mstsa

print(f"mstsa version: {mstsa.__version__}")

In [ ]:
# JupyterLite (cache-only) version: this copy ships only precomputed result
# caches (data/cache/...), not the raw MPILMBB dataset (158MB), so subject
# discovery via glob is skipped entirely.
data_path = "data/mpilmbb"  # not bundled in this copy; only used if a cache is missing
fs = 250          # Hz (4 ms / sample)
smoothing = 'b3'  # smoothing level
K = 4             # single clustering solution, no cross-K comparison
p_scale = dict(lmin=20, lmax=5000, fitmin=20, fitmax=5000, nsteps=30)
subjects = []     # cache-only: no raw subject list built (see note above)


## Fractional Gaussian noise (fGn) simulator

**Aim:**  
Explore fractional Gaussian noise (fGn) to get an idea how the mathematical prototype of a long-range dependent (LRD), fractal, scale-free stochastic process looks like and how `H` tunes these properties.
- `H=0.5`: uncorrelated white noise
- `0.5 < H < 1.0`: LRD with infinite autocorrelation; make the process as long as you like, it will never go flat, it will continue generating variance at every scale. Since the same statistical law generates the variance at every scale _by construction_, the process displays statistical self-similarity.

**Method:**  
`fGn(H, n)` synthesizes a fractional Gaussian noise trace with a prescribed Hurst
exponent `H` via the Davies & Harte (1987) circulant-embedding method.  
Note:  not part of `mstsa`

In [ ]:
def fGn(H, n):
    """Synthesize fractional Gaussian noise (fGn) with Hurst exponent H.

    0 < H <= 1: fractional Gaussian noise with Hurst index H.
    1 < H <= 2: fractional Brownian motion with Hurst index H-1 (the fGn
    increments are cumulatively summed before being returned).
    Either way, power spectral density is proportional to 1/f^(2H-1).

    Davies & Harte (1987) circulant-embedding method; adapted from `fGn` in
    `eeglib3.py` (FvW, 11/2013). See also Beran (1994); Bardet et al. (2002).

    Parameters
    ----------
    H : float
        Hurst exponent.
    n : int
        Number of samples.

    Returns
    -------
    y : ndarray, shape (n,)
        Fractional Gaussian noise (or fBm, if H > 1) sample.
    """
    if H > 1:
        H -= 1
    if H == 0.5:
        y = np.random.randn(n)
    else:
        nfft = int(2 ** np.ceil(np.log2(2 * (n - 1))))
        nfft2 = nfft // 2
        k = np.concatenate([np.arange(nfft2), np.arange(nfft2, 0, -1)])
        Zmag = 0.5 * ((k + 1) ** (2 * H) - 2 * k ** (2 * H) + np.abs(k - 1) ** (2 * H))
        Zmag = np.real(np.fft.fft(Zmag))
        if np.any(Zmag < 0):
            print("Negative values in the FFT of the circulant cov.")
        Zmag = np.sqrt(Zmag)
        Z = Zmag * (np.random.randn(nfft) + 1j * np.random.randn(nfft))
        y = np.real(np.fft.ifft(Z)) * np.sqrt(nfft)
        y = y[:n]
    if H > 1:
        y = np.cumsum(y)
    return y

In [ ]:
H_demo = 0.8      # Hurst exponent to simulate (0 < H <= 1 for fGn, 1 < H <= 2 for fBm)
n_demo = 15000     # number of samples

x_demo = fGn(H_demo, n_demo)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(x_demo, '-k', lw=0.8)
ax.set_xlabel('sample')
ax.set_ylabel('amplitude')
ax.set_title(f'Simulated {"fBm" if H_demo > 1 else "fGn"} trace, '
             f'H={H_demo}, n={n_demo}')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Global field power (GFP)

**Aim:**  
- Now, compare fGn with example GFP time courses from the LEMON database (eyes-closed resting-state EEG)
- The 16 recordings in the figure below all show multiple time scales across 1 minute of EEG, approximately.
- Variance is produced at many different time scales reaching the total length of the recording. Without any further knowledge about the statistics of the process, **H>0.5 in DFA can be predicted**.
- The observed fluctuations in total EEG power are most likely changes in vigilance and arousal levels (many people fall asleep during the resting-state), slow synaptic modulation etc.; each of these processes comes with its own biologically defined time scale. 
- The recordings below are **_scale-rich, not scale-free_**
- It seems plausible that these slow modulations are reflected in microstate dynamics as well, explaining H>0.5
- Assuming self-similarity - that the same process produces millisecond - second - minute fluctuations is a claim that DFA cannot prove as it only measures whether _something_ produces variance at each scale
- Although GFP timecourses are not the same as EEG microstate sequences, it wouldn't be a surprise if microstate dynamics co-varied with these fluctuations in brain electrical activity

The same alpha power / GFP comparison for 16 further example subjects (8 x 2 grid):

In [ ]:
# tile the 16 pre-rendered per-subject GFP/alpha-power figures into one grid
gfp_dir = "data/gfp_examples"
gfp_files = sorted(glob.glob(f"{gfp_dir}/Figure_gfp_*.png"))
n_cols_gfp = 2
n_rows_gfp = -(-len(gfp_files) // n_cols_gfp)  # ceil

fig, axes = plt.subplots(n_rows_gfp, n_cols_gfp, figsize=(14, 2.2 * n_rows_gfp))
for ax, f in zip(axes.flat, gfp_files):
    ax.imshow(plt.imread(f))
    ax.axis('off')
for ax in axes.flat[len(gfp_files):]:
    ax.set_visible(False)

fig.suptitle("Occipital alpha power (O1/O2 analytic amplitude) and GFP for 16 example subjects", y=1.0)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/05_occipital_alpha_gfp_examples_grid.png', dpi=150, bbox_inches='tight')
plt.show()


## Slow exponential dynamics?

A simple mechanism for persistence (H>0.5) is an exponentially (i.e. short-range) correlated process with a time constant that is large relative to the length of the recording.  

We investigated this mechanism in: von Wegner, F., Tagliazucchi, E., Brodbeck, V., & Laufs, H. (2016), "Analytical and empirical fluctuation functions of the EEG microstate random walk --
Short-range vs. long-range correlations", *NeuroImage*, 141, 442-451.

**Method:**  
- For `n_subjects_sample` randomly chosen subjects (fixed seed) and each of their `K`
microstates, `n_surr_slow` first-order Markov surrogates are synthesized matched to
that subject's own transition matrix and stationary distribution
(`mstsa.tpm_cond`/`mstsa.pmf`/`mstsa.mc_sample_path`), and the DFA Hurst exponent (`mstsa.hurst_exponent_dfa`) is computed for both the real sequence and
each surrogate.

**Interpretation:**
- The first-order Markov surrogates are memoryless beyond one step by construction,
their Hurst exponent reflects only whatever apparent scaling behavior that exponential time constant can produce.  
- The code applies a one-sided non-parametric test. Accepting the alternative hypothesis `H_real > H_surrogate`, might indicate genuine LRD, but also other mechanisms such as non-stationarity.  
- p-value: `p = (1 + #surrogates >= H_real) / (n_surr_slow + 1)` ; the smallest achievable p-value is `1/(n_surr_slow + 1)`, so `n_surr_slow` is set to 19.

In [ ]:
n_subjects_sample = 10
rng_slow = np.random.default_rng(0)
# JupyterLite (cache-only) version: sample_subjects is a placeholder here --
# the actual subject IDs used originally are stored in, and restored from,
# the cache below (72c4cbfe), which also fills in h_real_slow/h_surr_slow.
sample_subjects = []


In [ ]:
# partition-based DFA Hurst exponent (Van De Ville et al. 2010 / von Wegner et
# al. 2016): collapse the sequence into a +-1 random walk per bipartition
# (mstsa.partitions/mstsa.randomwalk) and take the DFA Hurst exponent of each
# walk's cumulative sum (mstsa.dfa) -- one H per bipartition, rather than one
# H per microstate as the indicator encoding (mstsa.hurst_exponents_dfa) gives.
# Stands in for the upcoming mstsa.hurst_exponent_dfa.
partitions_K = mstsa.partitions(K)
n_partitions = len(partitions_K)


def hurst_exponent_dfa(x, partitions, **p_scale):
    # mstsa.dfa performs its own integration (cumsum) internally, so it takes
    # the +-1 random walk INCREMENTS directly, not an already-cumulated walk
    return np.array([mstsa.dfa(mstsa.randomwalk(x, part), **p_scale)
                      for part in partitions])


n_surr_slow = 19  # smallest count giving a one-sided rank-test floor of 1/20 = 0.05

cache_dir_slow = "data/cache/slow_exponential_dynamics"
os.makedirs(cache_dir_slow, exist_ok=True)
cache_file_slow = os.path.join(
    cache_dir_slow,
    f"h_real_vs_surr_partitions_{smoothing}_K{K}_n{n_subjects_sample}_nsurr{n_surr_slow}_lmax{p_scale['lmax']}.npz")

if os.path.exists(cache_file_slow):
    d = np.load(cache_file_slow)
    h_real_slow = d['h_real_slow']
    h_surr_slow = d['h_surr_slow']
    n_samples_slow = d['n_samples_slow']
    sample_subjects = list(d['sample_subjects'])
    print(f"loaded cached results from {cache_file_slow}")
else:
    raise RuntimeError(
        "no cache found, and this JupyterLite copy does not bundle the raw "
        "MPILMBB dataset needed to recompute -- see ../notebooks/05_Hurst_phenomenon.ipynb"
    )

    h_real_slow = np.zeros((n_subjects_sample, n_partitions))
    h_surr_slow = np.zeros((n_subjects_sample, n_surr_slow, n_partitions))
    n_samples_slow = np.zeros(n_subjects_sample, dtype=int)

    for i, subj in enumerate(sample_subjects):
        fname = f"{data_path}/{subj}_EC_ms_{smoothing}_K{K}_ep00.npy"
        x = np.load(fname)
        nx = len(x)
        n_samples_slow[i] = nx

        h_real_slow[i] = hurst_exponent_dfa(x, partitions_K, **p_scale)

        # first-order Markov surrogate matched to this subject's own transition
        # matrix and stationary distribution
        T = mstsa.tpm_cond(x, K)
        p = mstsa.pmf(x, K)
        for j in range(n_surr_slow):
            surr = mstsa.mc_sample_path(T=T, n=nx, p=p)
            h_surr_slow[i, j] = hurst_exponent_dfa(surr, partitions_K, **p_scale)

        print(f"[{i+1}/{n_subjects_sample}] {subj}", end="\r")
    print()

    print("done.")
    np.savez(cache_file_slow, h_real_slow=h_real_slow, h_surr_slow=h_surr_slow,
             n_samples_slow=n_samples_slow, sample_subjects=np.array(sample_subjects))


In [ ]:
alpha_slow = 0.05

rows = []
pretty_rows = []
for i, subj in enumerate(sample_subjects):
    for k in range(n_partitions):
        real = h_real_slow[i, k]
        surr_vals = h_surr_slow[i, :, k]
        # one-sided rank-based surrogate test (Theiler et al.-style): is H_real
        # unusually HIGH relative to its own surrogates, not just "different"
        n_ge = int(np.sum(surr_vals >= real))
        p_one_sided = (1 + n_ge) / (n_surr_slow + 1)
        significant = bool(p_one_sided <= alpha_slow)
        ci_lo, ci_hi = np.percentile(surr_vals, [2.5, 97.5])

        rows.append(dict(subject=subj, state=k, H_real=real,
                          surr_mean=surr_vals.mean(), surr_std=surr_vals.std(ddof=1),
                          surr_max=surr_vals.max(), p_one_sided=p_one_sided,
                          significant=significant))
        pretty_rows.append({
            'subject': f'{subj} (P{k})',
            'H_real': f'{real:.3f}',
            'surrogate H, mean (95% CI)': f'{surr_vals.mean():.3f} ({ci_lo:.3f}; {ci_hi:.3f})',
            'p (one-sided)': f'{p_one_sided:.3f}{"*" if significant else ""}',
        })

df_slow = pd.DataFrame(rows)
df_pretty = pd.DataFrame(pretty_rows)
print(df_pretty.to_string(index=False))
print(f"\n* p<={alpha_slow} (one-sided rank test; minimum achievable p = "
      f"1/{n_surr_slow + 1} = {1 / (n_surr_slow + 1):.3f})")
print(f"{df_slow['significant'].sum()}/{len(df_slow)} (subject, state) pairs show "
      f"H_real significantly higher than the {n_surr_slow} first-order Markov surrogates")

print(f"\nHow often was each partition's H_real significantly higher than its "
      f"{n_surr_slow} first-order Markov surrogates, out of {n_subjects_sample} subjects:")
print(f"{'part':>6s} {'n_sig':>7s} {'fraction':>9s}")
for k in range(n_partitions):
    n_sig = int(df_slow.loc[df_slow['state'] == k, 'significant'].sum())
    print(f"{k:6d} {n_sig:4d}/{n_subjects_sample:<3d} {n_sig / n_subjects_sample:9.2%}")

In [ ]:
n_cols = 5
n_rows = -(-n_subjects_sample // n_cols)  # ceil
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3 * n_rows), sharey=True)
axes_flat = np.atleast_1d(axes).flatten()

for i, subj in enumerate(sample_subjects):
    ax = axes_flat[i]
    for k in range(n_partitions):
        ax.boxplot([h_surr_slow[i, :, k]], positions=[k], widths=0.5, showfliers=False,
                   patch_artist=True, boxprops=dict(facecolor='lightgray', alpha=0.7),
                   medianprops=dict(color='tab:blue'))
    ax.scatter(range(n_partitions), h_real_slow[i], color='tab:red', marker='D', s=40, zorder=5,
               label='real')
    ax.set_xticks(range(n_partitions))
    ax.set_xticklabels([f'P{k}' for k in range(n_partitions)])
    ax.set_title(f'{subj} ({n_samples_slow[i] / fs:.0f} s)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    if i == 0:
        ax.legend(fontsize=8)

for ax in axes_flat[n_subjects_sample:]:
    ax.set_visible(False)

axes_flat[0].set_ylabel('H (partition DFA)')

fig.suptitle(f'Real Hurst exponent vs. first-order Markov surrogates '
             f'(n_surr={n_surr_slow}, {n_subjects_sample} subjects, EC, {smoothing!r}, K={K})',
             y=1.02)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/slow_exponential_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations:**
- Depending on the sample you have chosen, you are likely to see 
    - some examples of Hurst exponents significantly larger than explained by the exponential (Markovian) surrogates
    - some examples where the exponential model is enough to reproduce the Hurst exponent
- This is a meaningful preliminary test to conduct before addressing more advanced properties

## Sojourn-time distributions - evidence for power-laws?

One way that a binary stochastic process can generate true LRD is when the states have power-law distributed sojourn times (microstate durations), 

$$
p(\tau) \sim \tau^{−\alpha}
$$

According to a theorem about stochastic processes, an alternating-renewal-process with such a sojourn time pdf, and 2 < α < 3 (finite mean, infinite variance), has an aggregated process (random walk) that converges to fractional Gaussian noise with
$$
H = (4 − α)/2.
$$
This only holds for 2 < α < 3 (giving H ∈ (0.5, 1).
This is sometimes written using the tail's survival exponent μ = α − 1, so H = (3 − μ)/2.

We can test whether empirical microstate sojourn times are power-law distributed and maybe generate persistence by that mechanism.  
The following distributions are considered:
- powerlaw
- exponential 
- stretched exponential (Weibull)
- lognormal

The code uses the amazing powerlaw package by Newman and Shalizi. Note that the plot shows the CCDF which has exponent μ = α − 1. 

**Procedure:**  
- For each microstate, sojourn times (durations) are pooled across all
`{len(subjects)}` subjects (`mstsa.sojourn_times_unordered`)
- a power-law is fitted and contrasted against three alternatives
    - exponential
    - stretched exponential (Weibull)
    - lognormal
- the contrast uses the log-likelihood-ratio test (`Fit.distribution_compare`): 
positive `R` favors the power law, negative `R` favors the alternative; `p` is 
the two-sided significance of that preference

In [ ]:
# JupyterLite/Pyodide only: the cache below stores pickled powerlaw.Fit
# objects, so powerlaw itself is still needed to unpickle them even on a
# cache hit. It's a pure-Python package, so a plain PyPI install works.
try:
    import piplite
    await piplite.install("powerlaw")
except ImportError:
    pass


In [ ]:
import pickle
import powerlaw as pl

alt_distributions = ('exponential', 'stretched_exponential', 'lognormal')

cache_dir_sojourn = "data/cache/sojourn_distribution_fit"
os.makedirs(cache_dir_sojourn, exist_ok=True)
cache_file_sojourn = os.path.join(cache_dir_sojourn, f"pooled_fits_ms_{smoothing}_K{K}.pkl")

if os.path.exists(cache_file_sojourn):
    with open(cache_file_sojourn, 'rb') as fh:
        fits = pickle.load(fh)
    print(f"loaded cached fits from {cache_file_sojourn}")
else:
    raise RuntimeError(
        "no cache found, and this JupyterLite copy does not bundle the raw "
        "MPILMBB dataset needed to recompute -- see ../notebooks/05_Hurst_phenomenon.ipynb"
    )

    # Pool sojourn-time durations per microstate across all subjects: mstsa extracts
    # each subject's own sojourns, but mstsa.sojourn_distribution_fit only fits a
    # single sequence at a time, so the cross-subject pooling happens here first.
    pooled_durations = [[] for _ in range(K)]
    for subj in subjects:
        fname = f"{data_path}/{subj}_EC_ms_{smoothing}_K{K}_ep00.npy"
        x = np.load(fname)
        durations_per_symbol = mstsa.sojourn_times_unordered(x, K)
        for k in range(K):
            pooled_durations[k].extend(durations_per_symbol[k])

    # Fit power law + alternatives to each state's pooled sojourn-time distribution,
    # via the same Clauset et al. (2009) MLE method mstsa.sojourn_distribution_fit
    # uses internally (the `powerlaw` package). Durations are converted from
    # samples to milliseconds (1 sample = 1000/fs ms) before fitting -- fs=250 Hz
    # makes this an exact x4 integer scaling, so the discrete fit is unaffected.
    fits = []
    for k in range(K):
        y = np.asarray(pooled_durations[k], dtype=float) * (1000.0 / fs)
        fit = pl.Fit(y, discrete=True, verbose=False)
        fits.append(fit)

    with open(cache_file_sojourn, 'wb') as fh:
        pickle.dump(fits, fh)

print(f"{'state':>6s} {'n_pooled':>9s} {'xmin (ms)':>9s} {'n_tail':>7s} {'alpha':>7s}")
for k in range(K):
    fit = fits[k]
    n_pooled = len(fit.data_original)
    n_tail = int(np.sum(fit.data_original >= fit.xmin))
    print(f"{k:6d} {n_pooled:9d} {fit.xmin:9.1f} {n_tail:7d} {fit.power_law.alpha:7.3f}")

print(f"\n{'state':>6s} {'alt. distribution':>22s} {'LLR (R)':>9s} {'p':>10s}  favors")
for k in range(K):
    fit = fits[k]
    for name in alt_distributions:
        R, p = fit.distribution_compare('power_law', name, normalized_ratio=True)
        favors = 'power law' if R > 0 else name.replace('_', ' ')
        print(f"{k:6d} {name:>22s} {R:+9.3f} {p:10.3g}  {favors}")


**Observations:**  
The results might depend on the sample but, usually, power-law wins against exponential (= non-Markovianity, again), but stretched exponential and lognormal fits win against power-law.

In [ ]:
def alpha_to_hurst(alpha):
    """Theoretical H implied by a power-law sojourn-time tail exponent alpha,
    via the Taqqu-Willinger-Sherman (1997) alternating-renewal-process result
    H = (4 - alpha) / 2, valid only for 2 < alpha < 3 (finite mean, infinite
    variance); returns nan outside that range rather than extrapolating."""
    if 2 < alpha < 3:
        return (4 - alpha) / 2
    return np.nan


dist_colors = {'power_law': 'tab:orange', 
               'exponential': 'tab:green',
               'stretched_exponential': 'tab:red', 
               'lognormal': 'tab:purple'}
dist_linestyles = {'power_law': '-', 
                   'exponential': '--',
                   'stretched_exponential': '-.', 
                   'lognormal': ':'}
dist_labels = {'power_law': 'power law', 
               'exponential': 'exponential',
               'stretched_exponential': 'stretched exp.', 
               'lognormal': 'lognormal'}

# Which fitted distributions to draw, in addition to the empirical CCDF; 
# e.g. drop 'exponential' from this list if its curve distorts the axes at 
# these scales.
dists_to_plot = ('power_law', 
                 #'exponential', 
                 'stretched_exponential', 
                 'lognormal')
alt_dists_to_plot = [name for name in alt_distributions if name in dists_to_plot]

fig, axes = plt.subplots(K, 1, figsize=(6, 4 * K), sharex=True)

for k, ax in enumerate(axes):
    fit = fits[k]
    alpha_k = fit.power_law.alpha
    h_pl_k = alpha_to_hurst(alpha_k)

    x_emp, ccdf_emp = fit.ccdf(original_data=True)
    ax.scatter(x_emp, ccdf_emp, s=25, facecolors='none', edgecolors='tab:blue',
               label='empirical', zorder=3)

    if 'power_law' in dists_to_plot:
        fit.power_law.plot_ccdf(ax=ax, color=dist_colors['power_law'],
                                 linestyle=dist_linestyles['power_law'],
                                 label=dist_labels['power_law'])
    for name in alt_dists_to_plot:
        getattr(fit, name).plot_ccdf(ax=ax, color=dist_colors[name],
                                      linestyle=dist_linestyles[name],
                                      label=dist_labels[name])

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_ylabel('CCDF')
    h_pl_str = f'{h_pl_k:.2f}' if np.isfinite(h_pl_k) else r'$\alpha \notin (2,3)$'
    ax.set_title(f'sojourn time fit (state {k}, alpha={alpha_k:.2f}, '
                 f'H_pl={h_pl_str})')
    ax.legend(fontsize=12, loc='lower left')
    ax.spines[['top', 'right']].set_visible(False)

axes[-1].set_xlabel('dwell time (ms)')

fig.suptitle(f'Population-level sojourn-time distribution fit '
             f'(EC, {smoothing}, K={K}, n=201 subjects)', y=1.005)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/sojourn_distribution_fits.png', dpi=150, bbox_inches='tight')
plt.show()

## Moving towards statistical self-similarity: diffusion entropy

- DFA only measures (detrended) variance at different time scales, but does not study the shape of the random walk density evolves over time
- In order to talk about statistical self-similarity, the density should scale with `H`
- The diffusion entropy algorithm (DEA, Scafetta & Grigolini, 2002) does exactly that, it tracks the density of the 'diffusing' microstate random walk

**Aim:**
- to test the density scaling relationship 
$$
f_1(x,t) = t^{-H} F(x/t^H).
$$


**Method:**
- rescale the microstate random walk `x(t)` by `t^{-H}` at several different `t`, and the whole family of marginal densities should collapse onto a single shape `F(u)`.
- at this stage, we will switch from partition-based microstate random walks to their indicator function

In [ ]:
cache_dir = "data/cache/hurst_vs_diffusion_entropy"
os.makedirs(cache_dir, exist_ok=True)
cache_file = os.path.join(cache_dir, f"hurst_dea_{smoothing}_K{K}_lmax{p_scale['lmax']}.npz")

if os.path.exists(cache_file):
    d = np.load(cache_file)
    hurst = d['hurst']
    delta = d['delta']
    print(f"loaded cached results from {cache_file}")
    n_subjects_hurst = len(hurst)
else:
    raise RuntimeError(
        "no cache found, and this JupyterLite copy does not bundle the raw "
        "MPILMBB dataset needed to recompute -- see ../notebooks/05_Hurst_phenomenon.ipynb"
    )

    hurst = np.zeros((len(subjects), K))
    delta = np.zeros((len(subjects), K))

    for i, subj in enumerate(subjects):
        fname = f"{data_path}/{subj}_EC_ms_{smoothing}_K{K}_ep00.npy"
        x = np.load(fname)

        hurst[i] = mstsa.hurst_exponents_dfa(x, **p_scale)
        delta[i] = mstsa.diffusion_entropy(x, **p_scale)

        print(f"[{i+1}/{len(subjects)}] {subj}", end="\r")
    print()

    print("done.")
    np.savez(cache_file, hurst=hurst, delta=delta)

### Comparison of DFA and DEA Hurst exponents

Before proceeding with Hurst exponents from Diffusion Entropy Analysis (DEA), 
let's verify that both yield comparable values of `H`.

In [ ]:
colors = plt.cm.viridis(np.linspace(0, 1, K))

h_pooled = hurst.flatten()
d_pooled = delta.flatten()
r_all, p_all = pearsonr(h_pooled, d_pooled)
rho_all, p_rho_all = spearmanr(h_pooled, d_pooled)
print(f"Pooled (n={len(h_pooled)} subject x microstate pairs): "
      f"Pearson r={r_all:.3f} (p={p_all:.2g}), Spearman rho={rho_all:.3f} (p={p_rho_all:.2g})")
print(f"Pooled mean H = {h_pooled.mean():.4f}, mean delta = {d_pooled.mean():.4f}, "
      f"mean(delta-H) = {(d_pooled - h_pooled).mean():+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
x_pos = np.arange(K)
width = 0.35
h_mean, h_sem = hurst.mean(axis=0), hurst.std(axis=0, ddof=1) / np.sqrt(n_subjects_hurst)
d_mean, d_sem = delta.mean(axis=0), delta.std(axis=0, ddof=1) / np.sqrt(n_subjects_hurst)
ax.bar(x_pos - width / 2, h_mean, width, yerr=h_sem, capsize=4, label='H (indicator DFA)', color='tab:blue')
ax.bar(x_pos + width / 2, d_mean, width, yerr=d_sem, capsize=4, label='delta (DEA)', color='tab:orange')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'MS {k}' for k in range(K)])
ax.set_ylabel('scaling exponent (mean +/- SEM)')
ax.set_title('Mean H vs. mean delta per microstate')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
lims = (min(h_pooled.min(), d_pooled.min()) - 0.02, max(h_pooled.max(), d_pooled.max()) + 0.02)
for k in range(K):
    ax.scatter(hurst[:, k], delta[:, k], s=10, alpha=0.4, color=colors[k], label=f'MS {k}')
ax.plot(lims, lims, 'k--', lw=1, label='delta = H')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.set_xlabel('H (indicator DFA)')
ax.set_ylabel('delta (DEA)')
ax.set_title(f'Pooled: r={r_all:.3f}, rho={rho_all:.3f}')
ax.legend(fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(f'Indicator Hurst exponent vs. diffusion entropy exponent, summary\n'
             f'(n={n_subjects_hurst} subjects, EC, smoothing={smoothing!r}, K={K})',
             fontsize=12, y=1.03)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/hurst_vs_dea_summary.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations:**
- Although the Hurst exponents are not identical, there is a good agreement in terms of a positive correlation between both methods.
- $H_\mathrm{DEA}$ is systematically smaller than $H_\mathrm{DFA}$

## The marginal density rescaling test

McCauley, Gunaratne & Bassler (2007) show that a process `x(t)` scales with exponent
`H` iff `x(t) =_d t^H x(1)` ("equal in distribution"), which forces the marginal
density itself into the scaling form
$$
f_1(x,t) = t^{-H} F(x/t^H).
$$
They also show this is a **necessary but not sufficient** condition for long-range dependence.

**Method:**  
For the indicator (characteristic microstate sequence) random walk `y(t) = cumsum(indicator_k(t) - mean)` (the same construction DFA operates on internally), draw increments
`dy = y(t0+t) - y(t0)` from many random start points `t0`, for several window lengths
`t` spanning the DFA fit range. Rescale each set of increments by `t^{-H}` using the
series' own DFA-estimated `H`, then overlay their histograms: collapse onto one curve
is consistent with density scaling; systematic spread or shape drift across `t` is
not. Laid out as a `K x 2` grid, one row per microstate:
- **left column**: the real microstate's own indicator random walk, rescaled by its own
  empirical DFA `H`. Tests whether that `H` corresponds to genuine density scaling
  or is just a fluctuation-function slope with no such structure behind it.
- **right column**: a synthetic fBm control -- cumulative sum of `fGn(H, n)` seeded
  with that *same* microstate's `H` -- expected to collapse by construction. Since
  every microstate has a different empirical `H`, each gets its own matched fBm
  analogue rather than sharing one reference `H` across rows.

In [ ]:
from scipy.stats import ks_2samp, gaussian_kde

def collect_rescaled_increments(y, H, ts, n_starts=300, seed=0):
    """Increments dy = y(t0+t) - y(t0) over random start points t0, rescaled by
    t^-H, for each window length in ts. If y scales with exponent H, the
    resulting distributions should be (approximately) the same across t."""
    rng = np.random.default_rng(seed)
    n = len(y)
    out = {}
    for t in ts:
        max_start = n - t
        size = min(n_starts, max_start)
        starts = rng.integers(0, max_start, size=size)
        incr = y[starts + t] - y[starts]
        out[t] = incr / (t ** H)
    return out


def plot_density(ax, data, color, label, style='hist', bins=40, n_grid=200):
    """Draw one rescaled-increment distribution as either a step histogram or a
    kernel density estimate, depending on `style` ('hist' or 'kde')."""
    if style == 'hist':
        ax.hist(data, bins=bins, density=True, histtype='step', color=color,
                 lw=1.5, label=label)
    elif style == 'kde':
        kde = gaussian_kde(data)
        xs = np.linspace(data.min(), data.max(), n_grid)
        ax.plot(xs, kde(xs), color=color, lw=1.5, label=label)
    else:
        raise ValueError(f"unknown style {style!r}, expected 'hist' or 'kde'")


density_plot_style = 'kde'  # 'hist' or 'kde'

ts_rescale = [50, 200, 800, 3200]  # window lengths (samples) within DFA fit range

# Real microstate random walks: all K microstates of one example subject, each 
# with its own DFA-estimated H, paired with its own matched fBm control (same H) 
# one H-matched fBm analogue per microstate

#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
example_subject = "sub-010297" 
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
cache_dir_rescale = "data/cache/hurst_density_rescaling"
os.makedirs(cache_dir_rescale, exist_ok=True)
cache_file_rescale = os.path.join(
    cache_dir_rescale, f"rescaling_{example_subject}_{smoothing}_K{K}.npz")

if os.path.exists(cache_file_rescale):
    d_cache = np.load(cache_file_rescale)
    h_example = d_cache['h_example']
    rescaled_real = {k: {t: d_cache[f'real_k{k}_t{t}'] for t in ts_rescale} for k in range(K)}
    rescaled_synth = {k: {t: d_cache[f'synth_k{k}_t{t}'] for t in ts_rescale} for k in range(K)}
    print(f"loaded cached rescaling-test results from {cache_file_rescale}")
else:
    raise RuntimeError(
        "no cache found, and this JupyterLite copy does not bundle the raw "
        "MPILMBB dataset needed to recompute -- see ../notebooks/05_Hurst_phenomenon.ipynb"
    )

    fname_example = f"{data_path}/{example_subject}_EC_ms_{smoothing}_K{K}_ep00.npy"
    x_example = np.load(fname_example)
    n_synth = len(x_example)  # match fBm control length to the real microstate sequence
    h_example = mstsa.hurst_exponents_dfa(x_example, **p_scale)

    rescaled_real = {}
    rescaled_synth = {}
    for k in range(K):
        indicator_k = (x_example == k).astype(np.float64)
        y_k = np.cumsum(indicator_k - indicator_k.mean())
        rescaled_real[k] = collect_rescaled_increments(y_k, h_example[k], ts_rescale)

        noise_k = fGn(h_example[k], n_synth)
        noise_k = noise_k - noise_k.mean()
        y_synth_k = np.cumsum(noise_k)
        rescaled_synth[k] = collect_rescaled_increments(y_synth_k, h_example[k], ts_rescale, seed=k)

    save_dict = {'h_example': h_example}
    for k in range(K):
        for t in ts_rescale:
            save_dict[f'real_k{k}_t{t}'] = rescaled_real[k][t]
            save_dict[f'synth_k{k}_t{t}'] = rescaled_synth[k][t]
    np.savez(cache_file_rescale, **save_dict)

fig, axes = plt.subplots(K, 2, figsize=(10, 3.2 * K), sharex=True)
colors_t = plt.cm.viridis(np.linspace(0, 1, len(ts_rescale)))

for k in range(K):
    ax_real, ax_synth = axes[k, 0], axes[k, 1]

    for t, color in zip(ts_rescale, colors_t):
        label = f't={t*1000/fs:.0f} ms'
        plot_density(ax_real, rescaled_real[k][t], color, label, style=density_plot_style)
        plot_density(ax_synth, rescaled_synth[k][t], color, label, style=density_plot_style)

    ax_real.set_ylabel('density')
    ax_real.set_title(f'{example_subject}, MS{k} (H_DFA={h_example[k]:.2f})', 
                      fontsize=10)
    ax_synth.set_title(f'matched fBm control (H={h_example[k]:.2f})', 
                       fontsize=10)
    ax_real.spines[['top', 'right']].set_visible(False)
    ax_synth.spines[['top', 'right']].set_visible(False)

axes[0, 1].legend(fontsize=8)
axes[-1, 0].set_xlabel('(y(t0+t) - y(t0)) / t^H')
axes[-1, 1].set_xlabel('(y(t0+t) - y(t0)) / t^H')

fig.suptitle('Density scaling - microstate (L) vs. H-matched fBm control (R)',
             y=1.005)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/hurst_density_rescaling_test.png', dpi=150, 
            bbox_inches='tight')
plt.show()

# Quantitative check: std of the rescaled increments should be approx. constant 
# across t if density scaling holds. 
# KS test between the shortest and longest t should not be rejected
'''
print(f"{'case':>18s} {'t':>6s} {'std(rescaled)':>14s}")
for k in range(K):
    for t in ts_rescale:
        print(f"{'MS' + str(k):>18s} {t:6d} {rescaled_real[k][t].std():14.4f}")
    for t in ts_rescale:
        print(f"{'MS' + str(k) + ' (fBm)':>18s} {t:6d} "
              f"{rescaled_synth[k][t].std():14.4f}")
'''

print()
for k in range(K):
    stat, p = ks_2samp(rescaled_real[k][ts_rescale[0]], rescaled_real[k][ts_rescale[-1]])
    print(f"MS{k}: KS test, t={ts_rescale[0]} vs t={ts_rescale[-1]} rescaled: "
          f"\nD={stat:.3f}, p={p:.3g}"
          f"{' (reject same distribution)' if p < 0.05 
             else ' (consistent with same distribution)'}\n")
    stat, p = ks_2samp(rescaled_synth[k][ts_rescale[0]], rescaled_synth[k][ts_rescale[-1]])
    print(f"MS{k} (fBm): KS test, t={ts_rescale[0]} vs t={ts_rescale[-1]} rescaled: "
          f"\nD={stat:.3f}, p={p:.3g}"
          f"{' (reject same distribution)' if p < 0.05 
             else ' (consistent with same distribution)'}\n")

**Conclusions:**
1. This is ongoing research into what self-similarity might mean for EEG microstate sequences.
2. It should be emphasized that this only refers to 'statistical self-similarity', i.e. similarity in distribution, not 'strict' self-similarity in the sense that microstate activation patterns are actually replayed at different time scales. In fGn/fBm, who have perfect statistical self-similarity, there is no 'replay' of structured patterns at different scales - everything is noise, but noise sampled from a density that shows scaling.